In [2]:
%load_ext autoreload
%autoreload 2


In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
from src.data_loader import load_and_merge_data, merge_oil_data
from src.features import build_macro_features, build_time_features
from src.train import run_baseline_training

# 1. Run full data engineering ingestion engine
df = load_and_merge_data()
df = merge_oil_data(df)
df = build_time_features(df)
df = build_macro_features(df)

# 2. Train baseline model and calculate initial score
model, features = run_baseline_training(df)


⏳ Loading train data...
🗓️ Repairing missing timeline gaps...
⏳ Loading store metadata...
🔗 Merging datasets...
% Optimizing memory usage...
✓ Data loaded successfully! Final shape: (3008016, 10)
⏳ Ingesting and treating oil prices...
✓ Oil price feature integrated cleanly.
🗓️ Engineering calendar and payday features...
🛢️ Engineering advanced oil price trends...
✂️ Splitting data into Train and Validation sets...


KeyError: "['sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_roll_mean_7'] not in index"

In [2]:
from src.train import run_baseline_training

# Run the training loop again. It will now automatically record the score!
model, features = run_baseline_training(df, experiment_name="initial_baseline")


✂️ Splitting data into Train and Validation sets...


KeyError: "['sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_roll_mean_7'] not in index"

In [3]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
from src.data_loader import load_and_merge_data, merge_oil_data
from src.features import build_macro_features, build_time_features, build_lag_features
from src.train import run_baseline_training

# 1. Pipeline execution including our history features
df = load_and_merge_data()
df = merge_oil_data(df)
df = build_time_features(df)
df = build_lag_features(df)    # <-- Injected history lags
df = build_macro_features(df)

# 2. Track this run in MLflow as a distinct experiment comparison step
model, features = run_baseline_training(df, experiment_name="added_historical_lags")


⏳ Loading train data...
🗓️ Repairing missing timeline gaps...
⏳ Loading store metadata...
🔗 Merging datasets...
% Optimizing memory usage...
✓ Data loaded successfully! Final shape: (3008016, 10)
⏳ Ingesting and treating oil prices...
✓ Oil price feature integrated cleanly.
🗓️ Engineering calendar and payday features...
⏳ Engineering structural historical lags...
✓ Time-series memory features generated cleanly.
🛢️ Engineering advanced oil price trends...
✂️ Splitting data into Train and Validation sets...
📊 Tracking execution under run: 'added_historical_lags'
🚀 Training LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.121788 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2149
[LightGBM] [Info] Number of data points in the train set: 2970594, number of used features: 21
[LightGBM] [Info] Start training from score 2.910

In [4]:
from src.evaluate import run_error_analysis

# This will print out exactly where your model is struggling!
val_analysis_df = run_error_analysis(df, model, features)


🔎 Extracting validation slices for error analysis...

⚠️ --- Top 5 Product Families with Highest Average Error ---
family
GROCERY I    564.911940
BEVERAGES    530.971665
CLEANING     333.381199
PRODUCE      254.313500
DAIRY         89.450144
Name: absolute_error, dtype: float64

🏢 --- Top 5 Stores with Highest Average Error ---
store_nbr
44    148.065437
40    136.008567
45    131.027620
47    128.497098
46    126.500886
Name: absolute_error, dtype: float64


C:\Users\adis2\Desktop\Python\Kaggle\store-sales-forecasting\src\evaluate.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_df.groupby("family")["absolute_error"]


In [3]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
from src.data_loader import load_and_merge_data, merge_oil_data
from src.features import build_lag_features, build_macro_features, build_time_features
from src.train import run_ensemble_training  # Updated import

# Execute full pipeline
df = load_and_merge_data()
df = merge_oil_data(df)
df = build_time_features(df)
df = build_lag_features(df)
df = build_macro_features(df)

# Run dual-model tracking block
lgb_model, xgb_model, features = run_ensemble_training(
    df, experiment_name="lgb_xgb_ensemble_blend"
)


⏳ Loading train data...
🗓️ Repairing missing timeline gaps...
⏳ Loading store metadata...
🔗 Merging datasets...
% Optimizing memory usage...
✓ Data loaded successfully! Final shape: (3008016, 10)
⏳ Ingesting and treating oil prices...
✓ Oil price feature integrated cleanly.
🗓️ Engineering calendar and payday features...
⏳ Engineering advanced group lag features...
🏢 Engineering store-type and product family target scales...
✓ Advanced scale features generated cleanly.
🛢️ Engineering advanced oil price trends...
✂️ Splitting data into Train and Validation sets...
🚀 Training LightGBM model component...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.131158 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2914
[LightGBM] [Info] Number of data points in the train set: 2970594, number of used features: 24
[LightGBM] [Info] Start training from

In [6]:
from src.data_loader import prepare_test_inference_data
from src.train import generate_kaggle_submission

# 1. Prepare clean data grid
inference_df, test_row_count = prepare_test_inference_data(df)

# 2. Process features sequentially
inference_df = build_time_features(inference_df)
inference_df = build_lag_features(inference_df)
inference_df = build_macro_features(inference_df)

# 3. Generate your fixed submission file
generate_kaggle_submission(inference_df, test_row_count, xgb_model, features)


⏳ Loading raw test dataset...
⏳ Loading store metadata for test set...
🔗 Stitching historical training tail to test grid for lag computations...
🗓️ Engineering calendar and payday features...
⏳ Engineering advanced group lag features...
🏢 Engineering store-type and product family target scales...
✓ Advanced scale features generated cleanly.
🛢️ Engineering advanced oil price trends...
🔮 Running inference on future test grid...
📋 Re-aligning predictions with raw Kaggle test format...
📊 Submission Row Count: 28512
📊 Expected Row Count: 28512
🎉 Pristine submission file saved successfully to: C:\Users\adis2\Desktop\Python\Kaggle\store-sales-forecasting\data\processed\submission.csv
